In [3]:
import json, os, torch, torch.nn as nn
from collections import Counter
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from models import TextLSTM, tokenize, encode, TEXT_LABELS

os.makedirs("saved", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
ds = load_dataset("dair-ai/emotion")
print(ds)

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

c:\Users\hp\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\datasets--dair-ai--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\hp\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\huggingface_hub\file_download.py

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [4]:
MAX_LEN = 40
counter = Counter(t for x in ds["train"]["text"] for t in tokenize(x))
vocab = {"<pad>": 0, "<unk>": 1}
for w, c in counter.items():
    if c >= 2:
        vocab[w] = len(vocab)
print("vocab size:", len(vocab))

def collate(batch):
    seqs = [torch.tensor(encode(b["text"], vocab, MAX_LEN)) for b in batch]
    lengths = torch.tensor([len(s) for s in seqs])
    x = pad_sequence(seqs, batch_first=True, padding_value=0)
    y = torch.tensor([b["label"] for b in batch])
    return x, lengths, y

train_dl = DataLoader(ds["train"], batch_size=64, shuffle=True, collate_fn=collate)
val_dl = DataLoader(ds["validation"], batch_size=128, collate_fn=collate)
test_dl = DataLoader(ds["test"], batch_size=128, collate_fn=collate)

vocab size: 7401


In [5]:
model = TextLSTM(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

def evaluate(dl):
    model.eval(); correct = total = 0; preds = []; ys = []
    with torch.no_grad():
        for x, l, y in dl:
            p = model(x.to(device), l).argmax(1).cpu()
            correct += (p == y).sum().item(); total += len(y)
            preds += p.tolist(); ys += y.tolist()
    return correct / total, ys, preds

best = 0
for epoch in range(10):
    model.train(); tot = 0
    for x, l, y in train_dl:
        opt.zero_grad()
        loss = loss_fn(model(x.to(device), l), y.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); tot += loss.item()
    acc, _, _ = evaluate(val_dl)
    print(f"epoch {epoch+1} loss {tot/len(train_dl):.4f} val_acc {acc:.4f}")
    if acc > best:
        best = acc
        torch.save(model.state_dict(), "saved/text_lstm.pth")

epoch 1 loss 1.5320 val_acc 0.5205
epoch 2 loss 1.0669 val_acc 0.7740
epoch 3 loss 0.6137 val_acc 0.8740
epoch 4 loss 0.3989 val_acc 0.8985
epoch 5 loss 0.2886 val_acc 0.9115
epoch 6 loss 0.2301 val_acc 0.9200
epoch 7 loss 0.1888 val_acc 0.9165
epoch 8 loss 0.1674 val_acc 0.9270
epoch 9 loss 0.1414 val_acc 0.9280
epoch 10 loss 0.1284 val_acc 0.9285


In [6]:
model.load_state_dict(torch.load("saved/text_lstm.pth", map_location=device))
acc, ys, preds = evaluate(test_dl)
print("test acc:", acc)
print(classification_report(ys, preds, target_names=TEXT_LABELS))

json.dump(vocab, open("saved/vocab.json", "w"))
json.dump({"max_len": MAX_LEN, "vocab_size": len(vocab)}, open("saved/text_config.json", "w"))

test acc: 0.923
              precision    recall  f1-score   support

     sadness       0.97      0.96      0.96       581
         joy       0.94      0.94      0.94       695
        love       0.77      0.84      0.80       159
       anger       0.93      0.92      0.92       275
        fear       0.87      0.92      0.90       224
    surprise       0.85      0.67      0.75        66

    accuracy                           0.92      2000
   macro avg       0.89      0.87      0.88      2000
weighted avg       0.92      0.92      0.92      2000

